# Data Assimilation with DART–CESM

Run a real forecast–assimilate–update cycle: a 3-member regional MOM6 ensemble in CESM,
updated every 24 hours (once each day) by the observations you made in Tutorial 1.

A typical DART-CESM experiment consists of the following steps:

1. Install CESM_DA.
2. (Re)generate the model domain.
3. Create a multi-instance (ensemble) CESM case.
4. Configure the case for assimilation.
5. Stage your observations and submit.
6. Examine what the assimilation did.

*This is Part 3 of the DART tutorial series:*   
[1. Working with Real Observations](tutorial1_real_observations.ipynb) ·
[2. Creating Synthetic Observations](tutorial2_synthetic_observations.ipynb) ·
**3. Data Assimilation with DART–CESM**

```{admonition} What you'll learn
:class: tip

- How DART runs inside CESM as the **ESP** (External System Processing) component,
  rather than having separate DA scripts.
- The cycling loop: forecast → `filter` → updated restarts → next forecast.   
  `filter` is DART's assimilation program.
- Multi-instance CESM: `ninst` = ensemble size.
- Two important ensemble DA controls in `&filter_nml`: **inflation** and
  **localization**, and where to set them (`user_nl_dart`).
- How to judge an assimilation: observation-space statistics and state-space increments.
```

```{admonition} What you'll produce
:class: important

A 3-cycle assimilation over 2023-06-15 → 2023-06-18: a running CESM case, `obs_seq.final`
files recording what happened to every observation, and increment maps showing where the
observations updated the ocean.
```

```{admonition} Prerequisites
:class: warning

- The [CrocoDash tutorial](../crocodash/tutorial.ipynb). You know how
  to build a regional MOM6 case.
- Observations from [Tutorial 1](tutorial1_real_observations.ipynb) in
  `<DART_DA_PROJECT_DIR>/obs/real/ocn_obs_seq` (or use the staged copy at `<CROC_DART_OBS>/real/ocn_obs_seq`).
- Derecho access and a project code.
```

# SECTION 1: Install the CESM DA fork

DART is integrated as the **ESP component** of CESM in CROCODILE. It is built and
called by CESM like any other component following the _setup, build, submit_ workflow.
For this tutorial, we'll setup the case in CrocoDash, set options using `./xmlchange`
and `user_nl_dart`, build the case with `./case.build`, and submit the case with 
`./case.submit`.

Clone the fork, check out the DA branch, and populate the
components:

```bash
git clone https://github.com/CROCODILE-CESM/CESM.git CESM_DA
cd CESM_DA/
git checkout full_regional_cesm_da
./bin/git-fleximod update
```

You will point `cesmroot` at this clone in Section 3. 

For more detail on the DART interface to CESM, such as what compsets are available with DA, 
see the [DART_interface documentation](https://crocodile-cesm.github.io/DART_interface/). For _much_ more detail on DART take a look at the [DART documentation](https://docs.dart.ucar.edu/en/latest/) or learn about research with DART at dart.ucar.edu.

# SECTION 2: The model domain

## Step 2.1: Set Up Your Experiment Parameters

Remember to match these with your observations from Tutorial 1. 

In [ ]:
# --- CROCODILE DART tutorial series parameters (same cell in all 3 notebooks) ---
from pathlib import Path
import datetime

DA_PROJECT_DIR = Path("<DART_DA_PROJECT_DIR>")        # your DA project working directory on Derecho

START = datetime.datetime(2023, 6, 15)   # must match RUN_STARTDATE in Tutorial 3
END   = datetime.datetime(2023, 6, 18)   # 3 days -> 3 one-day assimilation windows, centered on midnight
FREQ  = datetime.timedelta(hours=24)

# Bounding box:
LAT_MIN, LAT_MAX = 20.0, 25.0
LON_MIN, LON_MAX = -160.0, -155.0

OBS_TYPES = ["ARGO_TEMPERATURE", "ARGO_SALINITY"]

REAL_OBS_DIR      = DA_PROJECT_DIR / "obs" / "real" / "ocn_obs_seq"      # Tutorial 1 output
SYNTHETIC_OBS_DIR = DA_PROJECT_DIR / "obs" / "synthetic" / "ocn_obs_seq" # Tutorial 2 output

## Step 2.2: Regenerate the tutorial domain

Domain generation is taught in the
[CrocoDash tutorial](../crocodash/tutorial.ipynb). Here we just
run the three cells that build the same `Hawaii` domain, because creating a case
needs the live grid, topography, and vertical-grid objects.

In [ ]:
from CrocoDash.grid import Grid

grid = Grid(
  resolution = 0.05, # in degrees
  xstart = 200.0, # min longitude in [0, 360]
  lenx = 5.0, # longitude extent in degrees
  ystart = 20.0, # min latitude in [-90, 90]
  leny = 5.0, # latitude extent in degrees
  name = "hawaii",
)

In [ ]:
from CrocoDash.topo import Topo

topo = Topo(
    grid=grid,
    min_depth=9.5,  # in meters
)

In [ ]:
from pathlib import Path

bathymetry_path = Path("<GEBCO>")

if not bathymetry_path.exists():
    raise FileNotFoundError(
        "Bathymetry file not found, please replace with path to bathymetry file"
    )

topo.set_from_dataset(
    bathymetry_path=bathymetry_path,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
)

In [ ]:
topo.depth.plot()

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid = VGrid.hyperbolic(
    nk=75,  # number of vertical levels
    depth=topo.max_depth,
    ratio=20.0,  # target ratio of top to bottom layer thicknesses
)

# SECTION 3: Create a multi-instance CESM case

## Step 3.1: Set the case details

Make sure `cesmroot` points at the CESM_DA directory.

In [ ]:
from pathlib import Path
# CESM case (experiment) name
casename = "hawaii"

# CESM source root - this is your CESM_DA directory from Section 1.
cesmroot = "<CESM_DA>"

# Place where all your input files will be stored
inputdir = Path(DA_PROJECT_DIR) / "input_files" / casename

# CESM case directory
caseroot = Path(DA_PROJECT_DIR) / casename

print(f"Case:")
print(f" casename = {casename}")
print(f" cesmroot = {cesmroot}")
print(f" inputdir = {inputdir}")
print(f" caseroot = {caseroot}")

print(f"Observation dirctories - these will be used once you have created your case")
print(f" REAL_OBS_DIR = {REAL_OBS_DIR}")
print(f" SYNTHETIC_OBS_DIR = {SYNTHETIC_OBS_DIR}")


## Step 3.2: Create the case

One new concept compared to the CrocoDash tutorial is running multi-instance CESM. An ensemble filter 
needs an **ensemble**, that is a group of model forcasts. CESM runs `ninst` copies ("instances") of the model. 
In an assimilation experiment, each ensemble member (instance) starts from slightly different initial conditions. 
The spread of the ensemble members is what gives us information on model uncertainty. 

For more detail on ensemble data assimilation, see the DART [introduction to ensemble data assimilation](https://docs.dart.ucar.edu/en/latest/guide/introduction-ensemble-da.html).

Three members is workshop-sized, small enough to build and run in a tutorial session. Real ocean
DA experiments use 30–80 members, and `spin up` the oceans from different initial conditions to get ensemble spread. 
We will perturb the ensemble members in this tutorial to generate ensemble spread. 

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    ninst=3, # ensemble size: 3 instances of MOM6
    project="<PROJECT_CODE>",
    override=True,
    machine="derecho",
    compset="CR_JRA_DA", 
)

````{admonition} Global Alternative: raw create_newcase
:class: dropdown

If you are not using CrocoDash (for example, running CESM on a global grid), the case
can be created directly with CIME. `G_JRA_DA` is a global DA-enabled ocean compset:

```bash
./cime/scripts/create_newcase \
    --run-unsupported \
    --res TL319_t232 \
    --compset G_JRA_DA \
    --case $casedir \
    --ninst 3 \
    --multi-driver \
    --project <PROJECT_CODE>
```
````

````{admonition} Try it: why 3 instances and not 1?
:class: attention

Before reading on: what could DART's `filter` compute with a 3-member ensemble that it
could not compute with a single model run?
````

````{admonition} Answer
:class: dropdown

**Statistics**. The ensemble **spread** is the filter's estimate of forecast uncertainty, and
the ensemble **covariance** between an observed quantity and the model state is what turns
an observation of temperature at one point into corrections of salinity, currents, and
temperature nearby. With one member there is no spread and no covariance available. The
ensemble *is* the model uncertainty.

3 is a very small ensemble, so the covariance estimates are noisy. In practice, 30–80 members is
common, and the ensemble size is a key control on the quality of the assimilation.

````

## Step 3.3: Prepare forcing data

Exactly as in the CrocoDash tutorial: make sure the forcing covers the date range you are interested in.

In [ ]:
case.configure_forcings(
    date_range=["2023-06-15 00:00:00", "2023-06-18 00:00:00"],
    boundaries=["north", "south", "east", "west"],
    function_name="get_glorys_data_from_rda",
)

In [ ]:
case.process_forcings()

# SECTION 4: Configure the Case for Assimilation

We'll now swap to the terminal to use CESM's xmlchange command to query and set our experiment options.

## Step 4.1: Turn on data assimilation

XML settings turn a regional ocean case into a cycling DA experiment. Run these in a
terminal in your case directory (`caseroot` above):

```bash
cd <DART_DA_PROJECT_DIR>/hawaii/
./case.setup
```

We're using observations, so we need to set the CESM calendar to Gregorian rather than the default "no leap year" 
calendar.

```
./xmlchange CALENDAR=GREGORIAN
```

Turn on data assimilation for the ocean with:

```
./xmlchange DATA_ASSIMILATION_OCN=TRUE
```

Unlike a regular CESM run, we want to stop the model each day to assimilate observations.

```
# 3 one-day cycles, centered on midnight:
./xmlchange STOP_OPTION=ndays,STOP_N=1
./xmlchange DATA_ASSIMILATION_CYCLES=3

./case.setup --reset
```

```{admonition} The contract with Tutorial 1
:class: warning

`RUN_STARTDATE`, `STOP_OPTION`, `STOP_N`, must correspond to your observation files and your assimilation frequency. DART-CESM looks up one
obs_seq file per cycle by timestamp, and observations files that don't align are **silently
skipped**. If you changed `START` or `FREQ` in Tutorial 1, you will need to set `STOP_OPTION` and `STOP_N` accordingly. 
```

For this tutorial, we'll turn off short term archiving so our output files stay in the run directory. 

```
./xmlchange DOUT_S=FALSE
```

To confirm DART is wired in as the ESP component:

```bash
./xmlquery --partial DATA_ASS   # DATA_ASSIMILATION_* flags
./xmlquery --partial ESP        # ESP component should be DART
```

## 4.2 Set the DART namelist Options

The namelist options for dart are set in `user_nl_dart`. Note `user_nl_dart` applies to all active DA components.
If you are running DA with multiple components you may want to set [component specific namelist options](https://crocodile-cesm.github.io/DART_interface/running_dart_as_a_cesm_component.html#component-specific-namelist-options).
Since we are just running ocean assimilation, we'l use `user_nl_dart`.

To change DART settings, edit **`user_nl_dart`** in the case directory.

The three key namelists for ocean DA: `&model_nml`, `&obs_kind_nml`, and `&filter_nml`.


**`&model_nml`**: which MOM6 variables are in the state vector (temperature, salinity,
SSH, velocities are the default). Only state-vector variables are updated by the filter.

Edit `user_nl_dart` so it contains the following model options.  We're going to use pseudo depth as the vertical and 
we're going to have Salt, Temperature, U and V current in our state. 


```
&model_nml
use_pseudo_depth = .true. 
layer_name = 'Layer'
model_state_variables        = 'Salt ', 'QTY_SALINITY             ', 'NA', 'NA', 'UPDATE',
                               'Temp ', 'QTY_POTENTIAL_TEMPERATURE', 'NA', 'NA', 'UPDATE',
                               'u    ', 'QTY_U_CURRENT_COMPONENT  ', 'NA', 'NA', 'UPDATE',
                               'v    ', 'QTY_V_CURRENT_COMPONENT  ', 'NA', 'NA', 'UPDATE',
/
```

**`&obs_kind_nml`**: which observation types are used by filter:   
-  `assimilate_these_obs_types` - these observations **are allowed** to impact the model state.  
-  `evaluate_these_obs_types` - these observations **do not impact** the model state, but their forward operator is computed and
    recorded.

For this tutorial we'll assimilate ARGO_SALINITY and ARGO_TEMPERATURE. Add the following to `user_nl_dart` as well:

```
&obs_kind_nml
assimilate_these_obs_types = 'ARGO_SALINITY', 'ARGO_TEMPERATURE'
/
```

After assimilation the DART QC indicates whether an observation was evaluate only or used in the assimilation:
[DART outgoing quality control](https://docs.dart.ucar.edu/en/latest/assimilation_code/modules/assimilation/quality_control_mod.html#dart-outgoing-quality-control).

````{admonition} Try it: evaluate vs assimilate
:class: attention

In `user_nl_dart`, move `ARGO_SALINITY` from `assimilate_these_obs_types` to
`evaluate_these_obs_types`. What will change in `obs_seq.final`, and why might you run a
new observation type in evaluate mode before letting it change your ocean state?
````

````{admonition} Answer
:class: dropdown

Salinity observations still get forward-operator values recorded in `obs_seq.final`. You
can still compute their RMSE, but they no longer update the state; only temperature does.
Evaluate mode is a way to test model performance against independent observations without 
altering or updating the model's internal state.
````

**`&filter_nml`**. The ensemble filter itself:

For *much* more detail on DART options take a look at the 
[filter namelist documentation](https://docs.dart.ucar.edu/en/latest/assimilation_code/modules/assimilation/filter_mod.html#Namelist). The two main options for ensemble DA and inflation and localization. 

| Setting | Meaning |
|---|---|
| `inf_flavor`, `inf_initial` | **inflation**, grows ensemble spread to counter the overconfidence of small ensembles |
| `cutoff` | **localization** half-width in radians, limits how far one observation reaches. 0.02 rad ≈ 127 km at the equator |


### Generating Spread

If we submit the case as is, all three ensemble members will have the same initial conditions, the same forcing data, and output the same results.
This is no good for data assimilation, which requires some ensemble spread.  

Generating ensemble spread is an interesting scientific topic. Typically small random perturbations are added to each ensemble member, then the models are integrated until system can stabilize, reach physical balance, and provide a realistic spread of model states for data assimilation.  Alternative approaches in include running a model for a long time and creating an ensemble by using the state from the same day each year. For example, running for 40 years, and taking the 40 January 1st as though they occurred on the same year.

For this tutorial we will perturb from a single instance. Note a realistic data assimilation experiment would take time to spin up the model. 

Edit user_nl_dart to set filter_nml option `perturb_from_single_instance = .true.`

We're also going to write out the ensemble members at each step before assimilation "preassim". 

```
&filter_nml
perturb_from_single_instance = .true.
perturbation_method = 'model'
stages_to_write = 'output', 'preassim'
num_output_state_members = 3
output_members               = .true.
/
```

During the CESM build step, the namelist file DART will read is written to to `Buildconf/dartconf/input.nml.ocn`.  We can use `./preview_namelists` to check the file is created correctly:

```
./preview_namelists --comp esp
```

## 4.3 Check the Forcing Calendar

MOM6 expects the calendar for the input to be named "gregorian" rather than standard.  We'll enforce this using 

```
module load nco
cd inputdir/casename/ocnice
for f in *.nc; do
    ncatted -O -a calendar,time,o,c,"gregorian" "$f" "$f"
done
```


## 4.4 Using the cpudev Queue on Derecho 

Let's use the development queue to try running our case. Since we have a small domain, we'll reduce the number of tasks per instance. 

```
./xmlchange JOB_QUEUE=develop
./xmlchange NTASKS=2
./xmlchange ROOTPE_OCN=0
```

## Step 4.2 Tell CESM when your observations are

Unlike regular CESM components, DART also requires a list of observation files, to use in the assimilation. Tell CESM when your observations are, `<REAL_OBS_DIR>` from the experiment parameters cell in section 2.1

```
./xmlchange DART_OBS_ROOT="<REAL_OBS_DIR>"
```

```{admonition} Run an OSSE instead
:class: note

Swap real observations for synthetic observations from
[Tutorial 2](tutorial2_synthetic_observations.ipynb) and the same case becomes an OSSE:

`./xmchange DART_OBS_ROOT="<SYNTHETIC_OBS_DIR>"`

```
During the CESM build step, the list of observation files DART expects is written to to Buildconf/dart.input_data_list.  We can use `./preview_namelists` to check the file is created correctly:

```
./preview_namelists --comp esp
```

Check Buildconf/dart.input_data_list contains the files you expect.

```
cat Buildconf/dart.input_data_list
```

## Step 4.3: Build the case

```bash
qcmd -- ./case.build
```

The build takes a while. Let's take a look at how the assimilation will while we wait. 
The job will execute 3 cycles data assimilation since `DATA_ASSIMILATION_CYCLES=3`:

```text

               one cycle
            ┌──────────────────────────────────────────────────────────┐
            ▼                                                          │
 1. FORECAST      all 3 MOM6 instances advance 24 hours (1 day)        │
 2. FILTER        CESM's ESP layer calls DART filter:                  │
                    reads obs_seq.<window>.out + all 3 model states    │
                    computes the ensemble update                       │
                    writes diagnostics (obs_seq.final, *assim_mean.nc) │
 3. UPDATE        updated restart files replace the forecast restarts  │
 4. ADVANCE       CESM resubmits the next 24-hour segment ─────────────┘
```

During `case.build` CESM writes the `user_nl_dart` settings you made above out to a
read-only DART namelist at `Buildconf/dartconf/input.nml`, regenerated for each run of
DART. After the build completes, verify the namelist DART will actually use:

```
cat Buildconf/dartconf/input.nml.ocn
```

Check the file contains the `model_nml` and `obs_kind_nml` options you set above.

# SECTION 5: Submit the Data Assimilation Experiment

Submitting the case will run the `DATA_ASSIMILATION_CYCLES` in one jobs submission.

```bash
./case.submit
```

Each cycle, `filter` writes diagnostics into the run directory (file names carry the case
name, dart, component, and the cycle timestamp):

| File | Contents |
|---|---|
| `obs_seq.final` | Every observation with its prior (and optionally posterior) forward operator results, and DART QC value |
| `output_mean.nc` | Updated ensemble mean |
| `output_sd.nc` | Updated ensemble spread |

The files in your run directory will be called something like `hawaii.dart.ocn_obs_seq_final.2023-06-18-00000` which is
`{case}.dart.{component}_obs_seq_final.{timestamp}`.

Since we are also outputting state members there will be files for the individual ensemble members named *member_0001, *member_0002, *member_0003.


# SECTION 6: Examine the assimilation

State space diagnostics evaluate model variables and error covariances directly in the system's physical 
or model coordinate domain, whereas observation space diagnostics map model estimates to the measurement (observation) domain.

## Step 6.1: Observation-space diagnostics

`obs_seq.final` contains information about which observations were used in assimilation, and why any observations were not used.
It also contains the prior and optionally the posterior forward operator values for each observation and their mean and spread.

[pyDARTdiags](https://ncar.github.io/pyDARTdiags/) can be used to calculate and plot observation-space diagnostics from `obs_seq.final`.

We'll ook at used vs. rejected observations, and the prior and posterior statistics for an observation type.


In [ ]:
import pydartdiags.obs_sequence.obs_sequence as obsq
from pydartdiags.matplots import matplots as mp

# Your case run directory: cd caseroot && ./xmlquery --value RUNDIR
RUN_DIR = Path("<RUN_DIR>")

data_file = Path(RUN_DIR / "hawaii.dart.ocn_obs_seq_final.2023-06-18-00000")
obs_seq = obsq.ObsSequence(data_file)

obs_seq.possible_vs_used()

This will print out a table of the number of observations that were available (possible) and the number of observations that were used by filter.  This should print something like:

type	 | possible	| used
--|--|--
0	| ARGO_SALINITY	| 1481	|1481
1	| ARGO_TEMPERATURE	| 1484	| 1484

If you have observations that were not used, take a deeper dive into the [DART QC value](https://docs.dart.ucar.edu/en/latest/assimilation_code/modules/assimilation/quality_control_mod.html#dart-outgoing-quality-control).

Let's take a look at a profile plot, and what this is telling us about the assimilation. 
We'll take a look at the Argo temperature observations, binned at the depths below.

In [ ]:
type = 'ARGO_TEMPERATURE'
levels = [0.0, 100.0,  150.0, 200.0, 250.0, 300.0, 400.0, 500.0, 700, 850, 925, 1000] # depth in m

fig = mp.plot_profile(obs_seq, levels, type, bias=True, rmse=True, totalspread=True, depth=True)

### Interpreting the assimilation diagnostics

When you look at your plot, consider the following questions:

1. Did the assimilation change the bias?
Compare the prior and posterior bias curves. Where do they differ, and at what depths? A change in bias indicates that the assimilation has moved the ensemble mean relative to the observations. Ideally, assimilation should reduce the magnitude of the bias, although this may not occur everywhere.

2. Did the assimilation change the RMSE?
Compare the prior and posterior RMSE curves. Where does the RMSE increase or decrease? A decrease indicates better agreement between the ensemble and the observations at those depths. Also look at the overall RMSE reported in the statistics box.

3. Where does the assimilation have the largest effect?
Look for depths where the prior and posterior curves are most separated. These are regions where the observations are having the greatest impact on the analysis.

4. Does the impact occur where observations are available?
Compare the locations of the observations with changes in the prior and posterior curves. Do the largest changes occur near observed depths? If not, the assimilation may be spreading observational information to other depths through the ensemble covariance and localization.

5. What happened to the ensemble spread?
Compare the prior and posterior total spread. Does the spread decrease after assimilation? A decrease is expected because the observations provide additional information and constrain the ensemble.

6. Did the spread decrease everywhere, or only at particular depths?
Look at the shape of the prior and posterior spread curves. A localized reduction in spread can indicate where the observations are constraining the ensemble most strongly.

7. Did the reduction in spread correspond to an improvement in RMSE or bias?
This is an important question. A smaller posterior spread means the ensemble is more tightly clustered, but it does not necessarily mean that the analysis is more accurate. Compare the changes in spread with the changes in bias and RMSE.

8. Has the ensemble become too tightly constrained?
If the posterior spread is much smaller than the prior spread but the RMSE has not improved, this may indicate that the ensemble has become underdispersive. In other words, the ensemble is more confident without necessarily being more accurate.

9. What do the overall statistics tell you?
Compare the prior and posterior values in the Grand statistics box. Do the changes in the overall bias, RMSE, and total spread agree with what you see in the depth profiles?

```{admonition} This tutorial is a tiny number of ensemble members 
:class: important

This tutorial uses a small ensemble for computational convenience. Real experiments would typically use many more ensemble
members (e.g., 40–80). The results here are intended to illustrate how the assimilation works and how to interpret the 
diagnostics, not to support scientific conclusions about this particular dataset.

```


## Step 6.2: State-space diagnostics

Look at the model state before and after assimilation.

`filter` writes the ensemble mean state for each stage in `stages_to_write`. Comparing the mean
**before** assimilation (`preassim_mean.nc`) to the mean **after** assimilation (`output_mean.nc`)
gives the **increment**:

```text
increment = posterior mean - prior mean
```

Large increments show where the observations moved the model state. We'll plot the surface
`Temp` field for the last cycle (2023-06-18), pre- and post-assimilation, the increment between
them, and the locations of the observations that were actually assimilated.

In [ ]:
import xarray as xr

# Mean-state files carry the case name, stage, and cycle timestamp, same convention as
# obs_seq.final above. If the glob below finds nothing, check the actual file names with:
#   ls $RUNDIR/*preassim_mean* ; ls $RUNDIR/*output_mean*
preassim_file = sorted(RUN_DIR.glob("*preassim_mean*"))[-1]
output_file   = sorted(RUN_DIR.glob("*output_mean*"))[-1]

VAR   = "Temp"   # field to plot: one of the model_state_variables set in user_nl_dart
LEVEL = 0        # surface layer (Layer is the vertical dim set by layer_name in user_nl_dart)

preassim  = xr.open_dataset(preassim_file)[VAR].isel(Layer=LEVEL).squeeze()
posterior = xr.open_dataset(output_file)[VAR].isel(Layer=LEVEL).squeeze()
increment = posterior - preassim

# preassim_mean.nc / output_mean.nc carry no lon/lat coordinates, only grid-index
# dimensions (lonh/lath). ocean_geometry.nc is MOM6's static grid file and has the actual
# T-point geolon/geolat, so pull those in and attach them as coordinates for plotting.
geom_file = (sorted(RUN_DIR.glob("*ocean_geometry*")) or sorted(Path(".").glob("*ocean_geometry*")))[-1]
geom = xr.open_dataset(geom_file)
geolon = (("lath", "lonh"), geom["geolon"].values)
geolat = (("lath", "lonh"), geom["geolat"].values)

preassim  = preassim.assign_coords(geolon=geolon, geolat=geolat)
posterior = posterior.assign_coords(geolon=geolon, geolat=geolat)
increment = increment.assign_coords(geolon=geolon, geolat=geolat)

print(f"Preassim file: {preassim_file.name}")
print(f"Output file:   {output_file.name}")
print(f"Geometry file: {geom_file.name}")
print(f"Max |increment|: {float(abs(increment).max()):.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def lon180(lon):
    """Convert 0-360 longitudes (DART convention) to -180..180 for plotting."""
    lon = np.asarray(lon, dtype=float)
    return np.where(lon > 180, lon - 360, lon)

# geolon from ocean_geometry.nc is 0-360; convert to -180..180 to match LON_MIN/LON_MAX
# and the obs longitudes below. geolon/geolat are 2D (lath, lonh), so the replacement
# coordinate must be given as a (dims, data) tuple, not a bare array.
preassim  = preassim.assign_coords(geolon=(("lath", "lonh"), lon180(preassim["geolon"])))
posterior = posterior.assign_coords(geolon=(("lath", "lonh"), lon180(posterior["geolon"])))
increment = increment.assign_coords(geolon=(("lath", "lonh"), lon180(increment["geolon"])))

OBS_TYPE = "ARGO_TEMPERATURE"   # observations to overlay: 'ARGO_TEMPERATURE' or 'ARGO_SALINITY'
used_obs = obs_seq.select_used_qcs()
used_obs = used_obs[used_obs["type"] == OBS_TYPE]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={"projection": ccrs.PlateCarree()})
fields = [preassim, posterior, increment]
titles = ["Preassim (prior) mean", "Output (posterior) mean", "Increment (posterior - prior)"]
vmax = float(np.abs(increment).max())

for ax, da, title in zip(axes, fields, titles):
    is_increment = da is increment
    # geolon/geolat are the T-point coordinates from ocean_geometry.nc, degrees.
    da.plot(
        ax=ax, x="geolon", y="geolat", transform=ccrs.PlateCarree(),
        cmap="RdBu_r" if is_increment else "viridis",
        vmin=-vmax if is_increment else None,
        vmax=vmax if is_increment else None,
        cbar_kwargs={"shrink": 0.7, "label": VAR},
    )
    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.set_title(title)

# Overlay the observation locations that were actually assimilated.
axes[2].scatter(
    lon180(used_obs["longitude"]), used_obs["latitude"],
    s=20, edgecolor="k", facecolor="none", transform=ccrs.PlateCarree(),
    label=f"assimilated {OBS_TYPE}",
)
axes[2].legend(loc="lower left")

plt.tight_layout()
plt.show()

````{admonition} Try it: connect increment to observation
:class: attention

1. Find the largest temperature increment on the map. Overlay the assimilated observation
   locations from `used_obs` (watch the longitude convention. The model grid may be
   0–360). Is there an observation at the bull's-eye?
2. Repeat the plot for salinity (`"Salt"`): did *temperature* observations move the salt
   field? Why is that possible?
3. Localization: if you halved `cutoff` in `user_nl_dart` and reran, how would the
   footprint of each increment change?
````

````{admonition} Answer
:class: dropdown

1. There should almost always be an observation at or near a strong increment maximum.
2. Yes. The ensemble covariance between temperature and salinity carries the update
   across variables. That cross-variable transfer is the whole power (and risk) of
   ensemble DA with a small ensemble.
3. Increment footprints shrink toward the observation locations; corrections far from any
   observation disappear. Too small a cutoff wastes information, too large a cutoff allows
   spurious correlations to update the model state.

````

```{admonition} Single Observations Useful for Testing
:class: attention 

When implementing new observations or models a good step is to examine the impact
of [assimilating a single observation](https://docs.dart.ucar.edu/en/latest/guide/instructions-for-porting-a-new-model-to-dart.html#testing-localization-using-a-single-observation-and-an-idealized-ensemble). 

```

# Recap

```{admonition} What you learned
:class: tip

- A cycling DA experiment is a standard CESM case: DA compset + `ninst` members +
  `DATA_ASSIMILATION_OCN=TRUE`, submitted with `./case.submit`. DART cycles as the ESP
  component.
- `DATA_ASSIMILATION_CYCLES` is the number of assimilation cycles.
- `RUN_STARTDATE` and the cycle length are a **contract with your observation files**.
- DART is configured through `user_nl_dart`. The three key settings are which observation types to assimilate;
  inflation; and localization (`cutoff`).
- `obs_seq.final` can be used for observation-space diagnostics (RMSE,  totalspread, bias). 
- The difference between model state pre and post assimilation gives the state-space increments.
```

**Your takeaway:** a DA-enabled case you can rerun and reconfigure, plus
`obs_seq.final` and increment maps from your own 3-cycle experiment.

# Where to go from here?

- **Run longer**: set `END = 2023-09-30` in Tutorial 1, regenerate the observations, and
  increase `DATA_ASSIMILATION_CYCLES`.
- **Run an OSSE**: swap in synthetic observations from
  [Tutorial 2](tutorial2_synthetic_observations.ipynb).
- **More members**: raise `ninst` and examine how spread, inflation, and RMSE respond.
- **More observations**: add `GLIDER_*` or `BOTTLE_*` types in Tutorial 1. Start them in
  evaluate mode.
- Dive into the
  [DART documentation](https://docs.dart.ucar.edu), explore [pyDARTdiags](https://ncar.github.io/pyDARTdiags/) for observation space diagnostics, and try the Crocodile gallery's
  [mom6-tools page](../mom6_tools.md) to look the mean ensemble member.